In [ ]:
%pip install google-generativeai pandas


In [ ]:
dbutils.library.restartPython()



In [ ]:
import pandas as pd
import google.generativeai as genai

# Widget — paste your Gemini key in the box that appears at top
dbutils.widgets.text("gemini_key", "", "Gemini API Key")
api_key = dbutils.widgets.get("gemini_key")

# Load data
df = pd.read_csv("/Volumes/insight/default/titanic/Titanic.csv")

print(f"✅ Data loaded — {df.shape[0]} rows x {df.shape[1]} columns")
print(f"✅ API key received — {len(api_key)} characters")


In [ ]:
# Configure Gemini
genai.configure(api_key=api_key)
model = genai.GenerativeModel("gemini-flash-latest")

# Test call — simple question
response = model.generate_content("What is data science in 2 sentences?")

print("✅ Gemini API working!")
print()
print(response.text)


In [ ]:
def generate_insights(df, summary):
    """
    Sends dataset statistics to Gemini AI and
    returns business insights as a string.
    """
    prompt = f"""
You are a senior data scientist and business analyst.
Analyse this dataset and provide actionable business insights.

Dataset Overview:
- Rows                : {summary["rows"]}
- Columns             : {summary["columns"]}
- Column names        : {summary["column_names"]}
- Numeric columns     : {summary["numeric_columns"]}
- Categorical columns : {summary["categorical_columns"]}
- Missing values      : {summary["missing_values"]}
- Duplicate rows      : {summary["duplicates"]}

Statistical Summary:
{df.describe().to_string()}

Sample Data (first 5 rows):
{df.head(5).to_string()}

Please provide:
1. Dataset Overview      — what kind of data is this?
2. Key Findings          — 5 most important observations
3. Business Insights     — actionable recommendations
4. Data Quality Issues   — problems found and how to fix them
5. Suggested Next Steps  — what analysis should be done next

Be specific. Use actual numbers from the data.
"""
    response = model.generate_content(prompt)
    return response.text

print("✅ generate_insights() defined")


In [ ]:
# We need summary dict — define it here
def get_basic_summary(df):
    return {
        "rows"                : df.shape[0],
        "columns"             : df.shape[1],
        "column_names"        : list(df.columns),
        "dtypes"              : df.dtypes.astype(str).to_dict(),
        "missing_values"      : df.isnull().sum().to_dict(),
        "missing_percent"     : (df.isnull().sum() / len(df) * 100).round(2).to_dict(),
        "duplicates"          : int(df.duplicated().sum()),
        "numeric_columns"     : list(df.select_dtypes(include="number").columns),
        "categorical_columns" : list(df.select_dtypes(include="object").columns),
        "total_missing_cells" : int(df.isnull().sum().sum()),
        "memory_usage_kb"     : round(df.memory_usage(deep=True).sum() / 1024, 2),
    }

# Generate summary
summary = get_basic_summary(df)

# Call Gemini — this may take 5-10 seconds
print("Sending data to Gemini AI...")
print()

insights = generate_insights(df, summary)

print(insights)


In [ ]:
def answer_question(df, question):
    """
    Answers a natural language question about the dataset.
    """
    prompt = f"""
You are a data analyst. Answer this question about the dataset.

Dataset info:
- Shape    : {df.shape}
- Columns  : {list(df.columns)}
- Sample   : 
{df.head(10).to_string()}

Statistical summary:
{df.describe().to_string()}

Question: {question}

Give a direct, specific answer with exact numbers from the data.
"""
    response = model.generate_content(prompt)
    return response.text

print("✅ answer_question() defined")


In [ ]:
import time

# Test with 5 questions
questions = [
    "What is the average age of passengers?",
    "Which passenger class had the highest survival rate?",
    "What percentage of female passengers survived?",
    "Who paid the highest fare and did they survive?",
    "What are the top 3 insights from this dataset?"
]

for i, q in enumerate(questions, 1):
    print(f"Q{i}: {q}")
    print(f"A : {answer_question(df, q)}")
    print("-" * 60)
    
    # Wait 12 seconds between calls to respect free tier rate limit (5 requests/minute)
    if i < len(questions):
        time.sleep(12)


In [ ]:
%pip install pandas google-generativeai


In [ ]:
dbutils.library.restartPython()


In [ ]:
"""
InsightForge AI — 04_insights
==============================
Handles all Gemini AI interactions:
- generate_insights()    : full dataset analysis
- answer_question()      : single question answering
- ask_with_memory()      : multi-turn Q&A with history
- chat_session           : stateful conversation object

Author   : Palak Parihar
Platform : Databricks
Model    : gemini-flash-latest
"""

import pandas as pd
import google.generativeai as genai
from typing import Optional

# ── Widgets ───────────────────────────────────────────────────
dbutils.widgets.text(
    "dataset_path",
    "/Volumes/insight/default/titanic/Titanic.csv",
    "Dataset Path"
)
dbutils.widgets.text("gemini_key", "", "Gemini API Key")

DATASET_PATH = dbutils.widgets.get("dataset_path")
GEMINI_KEY   = dbutils.widgets.get("gemini_key")
GEMINI_MODEL = "gemini-flash-latest"

# ── Load data ─────────────────────────────────────────────────
df = pd.read_csv(DATASET_PATH)

# ── Configure Gemini ──────────────────────────────────────────
genai.configure(api_key=GEMINI_KEY)
model = genai.GenerativeModel(GEMINI_MODEL)

print("=" * 55)
print("  InsightForge AI — Insights Module")
print("=" * 55)
print(f"  Dataset  : {DATASET_PATH}")
print(f"  Shape    : {df.shape[0]} rows x {df.shape[1]} columns")
print(f"  Model    : {GEMINI_MODEL}")
print(f"  API key  : {len(GEMINI_KEY)} characters")
print("=" * 55)


In [ ]:
def build_dataset_context(df: pd.DataFrame) -> str:
    """
    Builds a compact dataset summary string for use in prompts.
    Called by every AI function so all prompts share the same
    consistent dataset description.

    This prevents repeating the same describe() code in
    every prompt and ensures all functions use the same format.

    Parameters
    ----------
    df : pd.DataFrame
        The dataset to summarise.

    Returns
    -------
    str : formatted context string ready to insert into any prompt
    """
    numeric_cols     = list(df.select_dtypes("number").columns)
    categorical_cols = list(df.select_dtypes("object").columns)
    missing_cols     = {
        col: int(df[col].isnull().sum())
        for col in df.columns
        if df[col].isnull().sum() > 0
    }

    context = f"""
DATASET CONTEXT
Shape              : {df.shape[0]} rows x {df.shape[1]} columns
Columns            : {list(df.columns)}
Numeric columns    : {numeric_cols}
Categorical columns: {categorical_cols}
Missing values     : {missing_cols if missing_cols else "None"}

STATISTICAL SUMMARY
{df.describe().round(2).to_string()}

SAMPLE DATA (first 5 rows)
{df.head(5).to_string()}
"""
    return context

# Test it
context = build_dataset_context(df)
print("✅ build_dataset_context() defined")
print(f"   Context length: {len(context):,} characters")
print()
print("Preview:")
print(context[:300] + "...")


In [ ]:
def generate_insights(df: pd.DataFrame) -> str:
    """
    Sends the full dataset context to Gemini and returns
    a structured 6-section business analysis.

    Uses build_dataset_context() for consistent formatting.
    Includes correlation analysis when target variable exists.

    Parameters
    ----------
    df : pd.DataFrame
        The cleaned dataset to analyse.

    Returns
    -------
    str : structured insights text with 6 labelled sections
    """
    context = build_dataset_context(df)

    prompt = f"""
You are a senior data scientist and business analyst.
Analyse this dataset and produce a professional report.

{context}

Write a structured analysis with EXACTLY these 6 sections.
Use actual numbers from the data. No generic statements.

1. EXECUTIVE SUMMARY
   3-4 sentences. What is this dataset? Most important finding?

2. KEY FINDINGS
   Exactly 5 bullet points, each with a specific number.
   Example: 74% of female passengers survived vs 19% of males.

3. BUSINESS INSIGHTS
   3-4 actionable recommendations for decision makers.

4. DATA QUALITY ASSESSMENT
   Comment on completeness, missing values, and data issues.

5. PATTERNS AND ANOMALIES
   Surprising relationships, outliers, or unusual distributions.

6. RECOMMENDED NEXT STEPS
   3 specific analyses or actions to take next.
"""

    try:
        response = model.generate_content(prompt)
        print(f"✅ Insights generated — {len(response.text):,} characters")
        return response.text
    except Exception as e:
        print(f"❌ generate_insights failed: {e}")
        return f"Error generating insights: {str(e)}"

print("✅ generate_insights() defined")


In [ ]:
def answer_question(df: pd.DataFrame, question: str) -> str:
    """
    Answers a single natural language question about the dataset.
    Each call is independent — no conversation history.
    Use ask_with_memory() for chained questions.

    Parameters
    ----------
    df       : pd.DataFrame — the dataset to query
    question : str — the user's question in plain English

    Returns
    -------
    str : direct answer with specific numbers from the data
    """
    context = build_dataset_context(df)

    prompt = f"""
You are a data analyst. Answer this question about the dataset.

{context}

Question: {question}

Rules:
- Give a direct, specific answer
- Include exact numbers where possible
- If the answer requires a calculation, show it
- Keep the answer under 200 words
- If you cannot answer from the data, say why
"""

    try:
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"Error answering question: {str(e)}"

print("✅ answer_question() defined")


In [ ]:
class ChatSession:
    """
    Manages a multi-turn conversation about a dataset.

    Maintains a rolling history of the last N exchanges
    so each new question has context from previous answers.
    This is what makes the Q&A feel like a real conversation
    rather than isolated single questions.

    Attributes
    ----------
    df              : the dataset being discussed
    history         : list of (question, answer) tuples
    max_history     : maximum number of exchanges to remember
    context         : pre-built dataset context string

    Example
    -------
    session = ChatSession(df)
    session.ask("What is the average age?")
    session.ask("How does that compare to survivors?")
    session.ask("What about by passenger class?")
    session.show_history()
    """

    def __init__(self, df: pd.DataFrame, max_history: int = 5):
        self.df          = df
        self.history     = []
        self.max_history = max_history
        self.context     = build_dataset_context(df)
        print(f"✅ ChatSession started")
        print(f"   Dataset shape    : {df.shape}")
        print(f"   Max history      : {max_history} exchanges")
        print(f"   Ask questions using session.ask('your question')")

    def ask(self, question: str) -> str:
        """
        Sends a question to Gemini with full conversation history.
        Automatically stores the exchange for future context.

        Parameters
        ----------
        question : str — the question to ask

        Returns
        -------
        str : Gemini's answer with dataset context and history
        """
        # ── Build history text ────────────────────────────────
        history_text = ""
        if self.history:
            history_lines = []
            # Only use last max_history exchanges
            recent = self.history[-self.max_history:]
            for i, (q, a) in enumerate(recent, 1):
                # Truncate long answers in history to save tokens
                short_a = a[:300] + "..." if len(a) > 300 else a
                history_lines.append(f"Q{i}: {q}")
                history_lines.append(f"A{i}: {short_a}")
            history_text = "\n".join(history_lines)

        # ── Build prompt with history ─────────────────────────
        prompt = f"""
You are a data analyst having a conversation about a dataset.
Use the conversation history for context when answering.

{self.context}

CONVERSATION HISTORY
{history_text if history_text else "This is the first question."}

CURRENT QUESTION
{question}

Rules:
- Answer the current question directly
- Reference previous answers when relevant
- Use specific numbers from the data
- Keep the answer under 250 words
- If the question builds on a previous answer, acknowledge it
"""

        try:
            response = model.generate_content(prompt)
            answer   = response.text

            # Store this exchange in history
            self.history.append((question, answer))

            return answer

        except Exception as e:
            error = f"Error: {str(e)}"
            self.history.append((question, error))
            return error

    def show_history(self):
        """Prints the full conversation history in a readable format."""
        if not self.history:
            print("No conversation history yet.")
            return

        print("=" * 55)
        print("  CONVERSATION HISTORY")
        print("=" * 55)
        for i, (q, a) in enumerate(self.history, 1):
            print(f"\n  Q{i}: {q}")
            print(f"  A{i}: {a[:400]}{'...' if len(a) > 400 else ''}")
            print(f"  {'─' * 50}")

    def reset(self):
        """Clears conversation history to start a fresh session."""
        count         = len(self.history)
        self.history  = []
        print(f"✅ Session reset — cleared {count} exchanges")

    def summary(self):
        """Prints a summary of the current session state."""
        print(f"  Session summary:")
        print(f"    Exchanges so far : {len(self.history)}")
        print(f"    Max history      : {self.max_history}")
        print(f"    Dataset          : {self.df.shape}")

print("✅ ChatSession class defined")


In [ ]:
# Test answer_question with 3 independent questions
questions = [
    "What is the average age of passengers?",
    "Which passenger class had the highest survival rate?",
    "What percentage of female passengers survived?"
]

print("Testing answer_question()...")
print("=" * 55)

for i, q in enumerate(questions, 1):
    print(f"\nQ{i}: {q}")
    answer = answer_question(df, q)
    print(f"A{i}: {answer}")
    print("─" * 55)


In [ ]:
# Create a chat session
session = ChatSession(df, max_history=5)

print()
print("Starting multi-turn conversation...")
print("=" * 55)

# Question 1 — standalone fact
q1 = "What is the average age of passengers?"
print(f"\nQ1: {q1}")
a1 = session.ask(q1)
print(f"A1: {a1}")
print("─" * 55)

# Question 2 — references Q1
q2 = "How does average age differ between survivors and non-survivors?"
print(f"\nQ2: {q2}")
a2 = session.ask(q2)
print(f"A2: {a2}")
print("─" * 55)

# Question 3 — builds further
q3 = "Which gender had the better survival rate and by how much?"
print(f"\nQ3: {q3}")
a3 = session.ask(q3)
print(f"A3: {a3}")
print("─" * 55)

# Question 4 — continues the thread
q4 = "Among female passengers specifically, did class matter for survival?"
print(f"\nQ4: {q4}")
a4 = session.ask(q4)
print(f"A4: {a4}")
print("─" * 55)

# Question 5 — synthesises everything
q5 = "Based on everything we discussed, who had the best and worst odds of survival?"
print(f"\nQ5: {q5}")
a5 = session.ask(q5)
print(f"A5: {a5}")
print("─" * 55)


In [ ]:
# Show the full conversation history
session.show_history()

# Show session summary
print()
session.summary()


In [ ]:
# Reset session for a fresh start
session.reset()

# Run an interactive session
# Change these questions to anything you want to ask
my_questions = [
    "How many passengers were on board and how many survived?",
    "What was the fare distribution — was it equal across classes?",
    "Based on the data, what single factor most determined survival?",
]

print("Interactive Q&A Session")
print("=" * 55)

for q in my_questions:
    print(f"\nYou: {q}")
    answer = session.ask(q)
    print(f"AI : {answer}")
    print("─" * 55)

print()
print(f"Session complete — {len(session.history)} exchanges recorded")


In [ ]:
# Generate full insights and save to your Volume
print("Generating full dataset insights...")
insights = generate_insights(df)

print()
print("=" * 55)
print("GEMINI INSIGHTS")
print("=" * 55)
print(insights)

# Save to Volume
output_path = "/Volumes/insight/default/titanic/insights.txt"
with open(output_path, "w") as f:
    f.write(insights)

print()
print(f"✅ Insights saved to {output_path}")
